# NB04 - LightGBM Training & Evaluation

---

## Purpose

Trains and evaluates the **single frozen LightGBM model** used as the scoring function for
all downstream conformal prediction notebooks. The model is trained once on the 1999-2003
observation period and evaluated on the calibration window and four subsequent test regimes. This notebook takes the finalized
loan-month modeling panel, fits one temporally disciplined LightGBM score function, stores
all required artefacts, and documents the exact conventions under which downstream CP
results must be interpreted.

Methodologically, the fitted score function is a gradient-boosted tree model: Friedman
(2001) is the foundational reference for gradient boosting as stagewise function
approximation, while Ke et al. (2017) is the LightGBM-specific reference for the efficient
GBDT implementation used here. The train/validation/evaluation design is chronological by observation month:
fitting uses 1999–2003 rows, 2004 is reserved for early stopping, and later
windows are used for evaluation. Because the target looks 12 months forward,
the split is not horizon-purged (Tashman, 2000; Kaufman et al., 2012).

The set-valued classification and ambiguity/efficiency interpretation itself is not estimated
in this notebook, but the downstream use of prediction sets follows the conformal and
set-valued classification literature (Vovk et al., 2005; Romano et al., 2020; Sadinle et al.,
2019).

---

## Notebook Structure

| § | Title | Description |
|---|---|---|
| **0** | Setup & Configuration | Dependencies, paths, manifest, feature classification |
| **1** | Data Loading | Copy training split to NVMe; two-pass lazy scan; temporal split |
| **2** | Class Imbalance & Datasets | `scale_pos_weight` from fitted rows; sequential LightGBM Dataset construction |
| **3** | Hyperparameter Search | Optuna TPE, 80 trials on 3% loan-level sample |
| **4** | Final Model Training | Full training set, early stopping, artefact save |
| **5** | Model Evaluation | AUROC/AP/Brier, subgroup AUROC, calibration tables, SHAP |
| **6** | Artifact Registry | `model_manifest.json` |

---

## Model Scope (as stated earlier)

The model predicts **transition from current or 30-DPD into 60+ DPD or REO Acquisition
within the next 12 months** for active loans. This is a transition-risk / discrete-time
event-prediction setup, not a general active-portfolio delinquency model. Loans already
at 60+ DPD at observation time are outside the base population by NB03 construction.
The loan-month panel representation is consistent with the discrete-time hazard
tradition, where event occurrence over an interval is represented as a binary outcome
on repeated period-level records (Singer & Willett, 1993; Shumway, 2001).

**Dataset scope:** As defined in NB01.

---

## Key Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| **Base learner** | LightGBM gradient-boosted decision trees | Gradient boosting is a stagewise additive function-approximation method (Friedman, 2001). LightGBM is an efficient GBDT implementation designed for large-scale tabular learning (Ke et al., 2017). |
| **Device** | `device="cpu"`, `N_OPTUNA_THREADS` threads | CPU was used for full-data training. |
| **Class imbalance** | `scale_pos_weight` from actual fitted rows | Correctly computed from the rows actually used for fitting (`obs_year < 2004`). This preserves the original row population, unlike undersampling, but it changes the weighted training objective and does **not** guarantee calibrated individual probabilities. LightGBM documentation gives the `scale_pos_weight`-specific probability warning; Niculescu-Mizil and Caruana (2005) are used only as broader calibration background for boosted-tree probability estimates and post-hoc calibration. |
| **Validation split** | Chronological 2004 holdout (`obs_year == 2004`) | Keeps 2004 observation rows out of fitting and preserves chronological model selection by observation month; the 12-month outcome horizon is not purged (Kaufman et al., 2012; Tashman, 2000). |
| **Effective training window** | `obs_year < 2004` (1999-2003) | `obs_year == 2004` is held out for early stopping; the manifest train split covers 1999-2004, but the fitted model uses only 1999-2003 rows. |
| **Hyperparameter search** | Optuna TPE, `N_OPTUNA_TRIALS` trials, deterministic `OPTUNA_SAMPLE_FRAC` loan-level sample | Optuna provides the hyperparameter-optimization and pruning framework (Akiba et al., 2019). The 3% loan-level sample and 80-trial budget are practical computational choices, not formal optimality guarantees. |
| **Tuning metric** | AUROC | AUROC is used as a threshold-free discrimination/ranking metric (Fawcett, 2006). Average Precision (AP) is also reported as a supplementary precision-recall metric informative under severe class imbalance (Davis & Goadrich, 2006); AP is computed via sklearn's step-function `average_precision_score`, which avoids the incorrect linear interpolation in PR space identified by Davis and Goadrich (2006). Because precision equals `TP / (TP + FP)`, cross-regime AP comparisons must be interpreted cautiously when base rates differ; that comparability caveat is an algebraic consequence of the precision definition, not a direct Davis-and-Goadrich theorem. |
| **Optuna pruner** | MedianPruner with configured startup/warmup settings | Prunes unpromising trials using intermediate validation AUC. The exact behavior follows the configured Optuna parameters in the code. Note: Akiba et al. (2019) demonstrate that ASHA outperforms MedianPruner in their experiments; MedianPruner is used here as a simpler alternative whose warmup and interval behaviour is more directly interpretable for a fixed sequential trial budget (Optuna Developers, n.d.). |
| **Final boosting rounds** | `num_boost_round=1000`, early stopping patience 25 | Allows enough boosting iterations while using the chronological 2004 validation set to select the best iteration. |
| **`max_bin`** | 255 in final training | Practical LightGBM CPU-training setting used in the final run. Do not interpret this as a universal CPU optimum. |
| **`min_child_samples` scaling** | Optuna value × fitted-row scale factor, floored at 5000 | Custom engineering convention. Because `min_child_samples` controls minimum leaf size, the sampled-data value is scaled to the full-data row count to avoid excessive leaf fragmentation; this is not a formal statistical theorem. |
| **Nonconformity score** | `s = 1 - p̂_y(x)` constructed downstream | Standard classification nonconformity-score logic for conformal prediction. The fitted LightGBM model is treated as a frozen score function before calibration/conformal thresholding; the split-calibration logic follows the general conformal prediction framework. The `s = 1 - p̂_y(x)` form corresponds to the least-ambiguous set-valued score that thresholds the estimated class probability (Sadinle et al., 2019), while classification-specific adaptive prediction sets are treated downstream (Vovk et al., 2005; Vovk, 2012; Romano et al., 2020). |
| **`loan_age` exclusion** | Excluded from active features | Duration variables are excluded to keep the score focused on cross-regime transport rather than seasoning mechanics; in conventional discrete-time hazard models they can be legitimate predictors (Singer & Willett, 1993; Shumway, 2001). See Feature Notes. |
| **Monotonic constraints** | Not applied | LightGBM supports monotone constraints, but they are not imposed here so the score function can express nonlinear feature interactions. This is a design choice, not a claim that unconstrained models are universally preferable. |
| **Within-loan correlation** | Not explicitly modeled in the LightGBM objective | Each loan contributes multiple observation-month rows. LightGBM treats rows as prediction observations rather than estimating inferential standard errors. Discrete-time hazard formulations likewise enter each period-level record as a separate observation and do not, on their own, correct for dependence among a unit's repeated records (Singer & Willett, 1993; Shumway, 2001); downstream conformal notebooks evaluate the consequences of temporal non-exchangeability and regime transport. |

---

## Feature Notes

All feature columns come from `panel_manifest.json["feature_cols"]`. See NBs 01-03 for detailed information on decisions regarding the usage of certain features.

---

## SCP, Mondrian, APS, and Probability Calibration

The downstream conformal evaluation uses the frozen LightGBM score function in different ways:

- **SCP and Mondrian CP** use a binary classification score of the form
  `s = 1 - p̂_y(x)` after downstream recalibration. Split conformal validity is obtained
  by calibrating scores on a held-out calibration set under the relevant exchangeability
  assumptions (Vovk et al., 2005). Mondrian CP applies the same idea within predefined
  groups to target group-conditional validity; the distinction between unconditional and
  conditional validity is discussed in Vovk (2012).

- **Subgroup AUROC is diagnostic, not a validity proof.** It measures whether the fitted
  score is discriminative inside a subgroup; formal Mondrian coverage depends on within-group
  calibration/exchangeability and quantile resolution (Fawcett, 2006; Vovk, 2012).

- **APS** uses cumulative probability mass rather than only a simple binary score. For
  classification, APS constructs prediction sets from ordered class-probability mass and
  then conformalizes this score; in binary classification, the absolute predicted
  probability can directly affect the adaptive score and hence set efficiency/class-wise
  behavior (Romano et al., 2020). This is why raw probability calibration matters
  operationally even though conformal marginal coverage is obtained by score calibration.

Because the model uses `scale_pos_weight`, raw probabilities may be miscalibrated; the
bin-based calibration tables, equal-width ECE (M = 10), and equal-frequency QECE document
this, and downstream isotonic recalibration addresses it before probability-mass-based APS
evaluation (Zadrozny & Elkan, 2002; Niculescu-Mizil & Caruana, 2005; Guo et al., 2017).

---

## Output of This Notebook

The notebook produces the frozen LightGBM booster and downstream manifest;
supporting tuning, evaluation, calibration, and SHAP diagnostics are documented
in the notebook. In
substantive terms, it establishes the exact **score-generating mechanism** whose behaviour
is later analysed in the conformal prediction notebooks under temporal shift, subgroup
structure, and calibration stress.

---
## Section 0 · Setup & Configuration

### 0.1 · Install Dependencies

In [ ]:
%pip install -q optuna polars pyarrow scikit-learn matplotlib optuna-integration[lightgbm] shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.4/833.4 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 162.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 171.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.7/613.7 kB 63.9 MB/s eta 0:00:00


### 0.2 · Imports

In [ ]:
from __future__ import annotations

import gc, json, os, shutil, sys, time, warnings
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import shap
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()

print(f"LightGBM : {lgb.__version__}")
print(f"Polars   : {pl.__version__}")
print(f"Optuna   : {optuna.__version__}")
print(f"SHAP     : {shap.__version__}")

LightGBM : 4.6.0
Polars   : 1.41.2
Optuna   : 4.9.0
SHAP     : 0.52.0


### 0.3 · Drive Mount & Path Configuration

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/master_thesis")

PANEL_DIR   = DRIVE_ROOT / "data"    / "panel"
MODELS_DIR  = DRIVE_ROOT / "models"
RESULTS_DIR = DRIVE_ROOT / "results"
SCORES_DIR  = RESULTS_DIR / "scores"
MANIFEST_DIR = DRIVE_ROOT / "manifests"

# Local NVMe: training split staging and model artefacts during training
LOCAL_ROOT       = Path("/content") if IS_COLAB else DRIVE_ROOT
LOCAL_TRAIN_DIR  = LOCAL_ROOT / "panel_train"    # copy of split=train
LOCAL_MODELS_DIR = LOCAL_ROOT / "models_local"   # staging before Drive sync

for d in [MODELS_DIR, SCORES_DIR, MANIFEST_DIR,
          LOCAL_TRAIN_DIR, LOCAL_MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert PANEL_DIR.exists(), f"Panel directory not found: {PANEL_DIR}"
splits = sorted(p.name for p in PANEL_DIR.iterdir() if p.is_dir())
print(f"DRIVE_ROOT : {DRIVE_ROOT}")
print(f"PANEL_DIR  : {PANEL_DIR}")
print(f"Splits     : {splits}")

Mounted at /content/drive
DRIVE_ROOT : /content/drive/MyDrive/master_thesis
PANEL_DIR  : /content/drive/MyDrive/master_thesis/data/panel
Splits     : ['split=calibration', 'split=test_covid', 'split=test_normal', 'split=test_rate_hike', 'split=test_subprime', 'split=train']


### 0.4 · Load Panel Manifest

`panel_manifest.json` from NB03 is the authoritative contract for this notebook.

In [ ]:
manifest_path = MANIFEST_DIR / "panel_manifest.json"

with open(manifest_path) as f:
    MANIFEST = json.load(f)

FEATURE_COLS             = MANIFEST["feature_cols"]
CANDIDATE_COLS           = MANIFEST.get("candidate_feature_cols", [])
EXCLUDED_TRAIN_CONSTANT  = MANIFEST.get("excluded_training_constant_feature_cols", [])
HARD_EXCLUDED_FEATURES   = MANIFEST.get("hard_excluded_model_feature_cols", [])
SUBGROUP_COLS            = MANIFEST["subgroup_cols"]
META_COLS                = MANIFEST.get("meta_cols", [])
SPLIT_WINDOWS            = MANIFEST["split_windows"]
ALPHA                    = MANIFEST["alpha"]
SPLIT_STATS              = {row["split"]: row for row in MANIFEST["split_statistics"]}
TARGET_LABEL             = MANIFEST.get("target_label", "60+ DPD or REO Acquisition")
# Target column name is governed by the panel manifest contract.
TARGET_COL               = MANIFEST["target_col"]
assert TARGET_COL == "y", (
    f"Panel manifest target_col is '{TARGET_COL}', but the column references in this "
    f"notebook assume 'y'. Update those references to use TARGET_COL before proceeding."
)

print(f"Manifest loaded - created: {MANIFEST.get('created_at','?')}")
print(f"  Target label               : {TARGET_LABEL}")
print(f"  Active model features      : {len(FEATURE_COLS)}")
print(f"  Candidate features         : {len(CANDIDATE_COLS)}")
print(f"  Excluded train-constants   : {len(EXCLUDED_TRAIN_CONSTANT)}")
print(f"  Hard exclusions (upstream) : {len(HARD_EXCLUDED_FEATURES)}")
print(f"  Subgroup columns           : {SUBGROUP_COLS}")
print(f"  Alpha                      : {ALPHA}")
print()

for split in ["train", "calibration", "test_subprime", "test_normal", "test_covid", "test_rate_hike"]:
    s = SPLIT_STATS.get(split, {})
    print(f"  {split:<18}: {int(s.get('n_rows',0)):>13,} rows  pos={s.get('positive_rate_pct',0):.4f}%")

print("\nExcluded training-constant features (not passed to LightGBM):")
for col in EXCLUDED_TRAIN_CONSTANT:
    print(f"  - {col}")

print("\nHard-excluded model features retained only as panel diagnostics / metadata:")
for col in HARD_EXCLUDED_FEATURES:
    print(f"  - {col}")

# NB04 assumes NB03 already removed these from the active feature set.
for forbidden in ["channel", "deferred_upb_ratio", "in_workout_plan_flag"]:
    assert forbidden not in FEATURE_COLS, (
        f"Manifest contract violated: '{forbidden}' must not be in active model features."
    )

# test_normal must be the trimmed clean pre-COVID window from NB03
assert SPLIT_WINDOWS["test_normal"][1] == "2019-09-30", (
    "Manifest contract violated: test_normal must end at 2019-09-30."
)

Manifest loaded - created: 2026-06-19T16:27:17.483766+00:00
  Target label               : 60+ DPD or REO Acquisition
  Active model features      : 29
  Candidate features         : 30
  Excluded train-constants   : 1
  Hard exclusions (upstream) : 8
  Subgroup columns           : ['fico_tier', 'ltv_bucket', 'fico_ltv_group', 'census_division']
  Alpha                      : 0.1

  train             :   248,091,644 rows  pos=1.1560%
  calibration       :   179,344,121 rows  pos=1.1075%
  test_subprime     :   633,244,002 rows  pos=2.5252%
  test_normal       :   753,732,904 rows  pos=1.5173%
  test_covid        :   201,447,058 rows  pos=1.6793%
  test_rate_hike    :   306,340,880 rows  pos=1.0427%

Excluded training-constant features (not passed to LightGBM):
  - dti_missing_relief_refi

Hard-excluded model features retained only as panel diagnostics / metadata:
  - channel
  - modification_flag
  - payment_deferral_flag
  - loan_age
  - remaining_months_to_legal_maturity
  - amortiza

### 0.5 · Feature Classification

The active model feature set comes **exclusively** from `manifest["feature_cols"]`.  
This notebook only classifies already-approved features into LightGBM type categories
(string categorical, integer categorical, numeric). It never adds or removes features
from the active list.

**String categoricals** are integer-encoded using encoders fitted on the fitted training
data. Unseen categories at test time are mapped to a reserved unknown code
(`max_training_code + 1`). This is a reproducibility convention for out-of-time prediction,
not a separate statistical model.

**Integer categoricals** are passed to LightGBM as-is with the `categorical_feature`
designation. LightGBM supports categorical-feature handling for integer-coded categorical
features, so one-hot encoding is not required here (LightGBM Developers, n.d.).

**Numeric features** (including binary flags and missingness indicators) require no special
encoding.

In [ ]:
# String categoricals: label-encoded on training data; unseen values mapped to unknown code.
CAT_FEATURES_STR = [
    "loan_purpose",
    "occupancy_status",
    "property_type",
    "first_time_homebuyer_flag",
    "census_division",
]

# Integer categoricals: passed to LightGBM as-is
# number_of_borrowers is NOT included - replaced by is_multi_borrower (binary, numeric)
# vintage_quarter is 1-4 (ordinal intent but treated as categorical)
CAT_FEATURES_INT = [
    "vintage_year",
    "vintage_quarter",
    "number_of_units",
]

FEATURE_COLS_SET   = set(FEATURE_COLS)
cat_feature_names  = [c for c in CAT_FEATURES_STR + CAT_FEATURES_INT if c in FEATURE_COLS_SET]
numeric_feat_names = [c for c in FEATURE_COLS if c not in set(cat_feature_names)]

# Columns loaded for model fitting (train and validation passes)
TRAIN_LOAD_COLS = FEATURE_COLS + ["y", "obs_year"]

# Columns loaded for evaluation (adds subgroups and workout-plan metadata)
# in_workout_plan_flag is metadata for COVID-era subgroup analysis only;
# it is never passed to predict()
EVAL_LOAD_COLS = list(dict.fromkeys(
    FEATURE_COLS + ["y"] + SUBGROUP_COLS +
    (["in_workout_plan_flag"] if "in_workout_plan_flag" in META_COLS else [])
))

print(f"Active model features : {len(FEATURE_COLS)}")
print(f"  String categoricals : {[c for c in CAT_FEATURES_STR if c in FEATURE_COLS_SET]}")
print(f"  Integer categoricals: {[c for c in CAT_FEATURES_INT if c in FEATURE_COLS_SET]}")
print(f"  Numeric features    : {len(numeric_feat_names)}")
print(f"  Eval load columns   : {len(EVAL_LOAD_COLS)}")

# Verify no excluded-constant feature sneaked back into the active list
overlap = sorted(set(EXCLUDED_TRAIN_CONSTANT).intersection(FEATURE_COLS_SET))
assert len(overlap) == 0, (
    f"Manifest contract violated: training-constant features in active list: {overlap}"
)

# Verify hard-excluded policy/regime features did not sneak into the active list
hard_overlap = sorted(set(HARD_EXCLUDED_FEATURES).intersection(FEATURE_COLS_SET))
assert len(hard_overlap) == 0, (
    f"Manifest contract violated: hard-excluded features in active list: {hard_overlap}"
)

print("\n✓  No training-constant features in active model feature list.")
print("✓  No hard-excluded policy/regime features in active model feature list.")

Active model features : 29
  String categoricals : ['loan_purpose', 'occupancy_status', 'property_type', 'first_time_homebuyer_flag', 'census_division']
  Integer categoricals: ['vintage_year', 'vintage_quarter', 'number_of_units']
  Numeric features    : 21
  Eval load columns   : 34

✓  No training-constant features in active model feature list.
✓  No hard-excluded policy/regime features in active model feature list.


---
## Section 1 · Data Loading

### Memory Strategy - Two-Pass Lazy Scan on Local NVMe

The training split (`obs_year < 2004` plus `obs_year == 2004`) is copied from Drive to local
NVMe before data loading. This is a practical I/O and memory-management choice for repeated Optuna and full-training passes.

The notebook uses a two-pass lazy-scan strategy:

1. load only fitted training rows (`obs_year < 2004`);
2. load only validation rows (`obs_year == 2004`).

This avoids materializing unnecessary rows and preserves the chronological
observation-time split. The holdout keeps 2004 observation rows out of fitting;
it does not purge overlap in the 12-month forward outcome horizon.

In [ ]:
# ── Copy training split Drive → local NVMe (once per session) ─────────────────
if list(LOCAL_TRAIN_DIR.glob("*.parquet")):
    n_local = len(list(LOCAL_TRAIN_DIR.glob("*.parquet")))
    print(f"⏭  Training split already on NVMe ({n_local} files) - skipping copy.")
else:
    src = PANEL_DIR / "split=train"
    assert src.exists(), f"Training split not found: {src}"
    print("Copying training split Drive → local NVMe ...")
    t0 = time.time()
    for f in sorted(src.glob("*.parquet")):
        shutil.copy2(f, LOCAL_TRAIN_DIR / f.name)
    elapsed = time.time() - t0
    size_gb = sum(f.stat().st_size for f in LOCAL_TRAIN_DIR.glob("*.parquet")) / 1e9
    print(f"✓  Copied {size_gb:.2f} GB")

train_files = sorted(LOCAL_TRAIN_DIR.glob("*.parquet"))
print(f"Training files on NVMe: {len(train_files)}")

Copying training split Drive → local NVMe ...
✓  Copied 6.38 GB
Training files on NVMe: 1


### Categorical Encoding

A single encoder is fitted per **string-categorical** feature on the fitted
training subset (`obs_year < 2004`) and reused unchanged for validation,
calibration, and all test splits.

NB03 already converts field-specific raw sentinels to semantic categories before
NB04. In particular, `first_time_homebuyer_flag` maps raw `'9'` to `Unknown`
and blank/`NULL` to `Not_Applicable`; NB04 therefore encodes the cleaned panel
categories rather than preserving those raw representations.

Observed training categories receive their fitted integer codes. Any category
first appearing later is mapped to the reserved `max_training_code + 1` code,
preserving one stable train-time encoding contract across periods.

In [ ]:
CAT_ENCODERS: dict[str, LabelEncoder] = {}
UNKNOWN_CAT_CODES: dict[str, int] = {}

def _cat_values_for_encoding(s: pl.Series) -> np.ndarray:
    """
    Convert a cleaned string-categorical series to a stable string array.
    NB03 has already mapped field-specific raw sentinels to semantic categories;
    `__NULL__` is only a fallback for any actual null that remains.
    """
    return s.cast(pl.Utf8).fill_null("__NULL__").to_numpy().astype(str)

def fit_cat_encoders(df: pl.DataFrame) -> None:
    """Fit one LabelEncoder per string categorical column on the fitted training subset."""
    global CAT_ENCODERS, UNKNOWN_CAT_CODES
    CAT_ENCODERS = {}
    UNKNOWN_CAT_CODES = {}

    for col in CAT_FEATURES_STR:
        if col not in df.columns:
            continue

        le = LabelEncoder()
        vals = _cat_values_for_encoding(df[col])
        le.fit(vals)

        CAT_ENCODERS[col] = le
        UNKNOWN_CAT_CODES[col] = int(len(le.classes_))

        # Legacy diagnostic only: NB03 already maps raw FTHB NULL/blank and code '9'
        # to semantic categories before NB04, so neither raw representation is expected here.
        if col == "first_time_homebuyer_flag":
            has_null = "__NULL__" in le.classes_
            has_9 = "9" in le.classes_

            if not has_null or not has_9:
                print(
                    f"   Note: {col} fitted without "
                    f"{'__NULL__' if not has_null else ''}"
                    f"{' and ' if (not has_null and not has_9) else ''}"
                    f"{'9' if not has_9 else ''}. "
                    "This is acceptable if those categories are absent from the "
                    "obs_year < 2004 fitting window; later unseen values will be "
                    "mapped to the reserved unknown category."
                )

    print("✓  LabelEncoders fitted:")
    for col, le in CAT_ENCODERS.items():
        unk = UNKNOWN_CAT_CODES[col]
        has_null = "__NULL__" in le.classes_
        has_9 = "9" in le.classes_
        print(
            f"   {col:<28} classes={len(le.classes_):>3}  "
            f"unknown_code={unk:>3}  has___NULL__={has_null}  has_'9'={has_9}"
        )

def encode_cats(df: pl.DataFrame) -> pl.DataFrame:
    """
    Apply fitted encoders to string categoricals.
    Training-seen categories keep their fitted codes.
    Unseen categories map to a reserved unknown code = n_classes.
    """
    if not CAT_ENCODERS:
        raise RuntimeError("Call fit_cat_encoders() before encode_cats().")

    for col, le in CAT_ENCODERS.items():
        if col not in df.columns:
            continue

        vals = _cat_values_for_encoding(df[col])
        class_to_code = {cls: i for i, cls in enumerate(le.classes_)}
        unk_code = UNKNOWN_CAT_CODES[col]

        codes = np.fromiter(
            (class_to_code.get(v, unk_code) for v in vals),
            dtype=np.int32,
            count=len(vals),
        )

        df = df.with_columns(pl.Series(name=col, values=codes, dtype=pl.Int32))

    return df

print("✓  Categorical encoder helpers defined.")

✓  Categorical encoder helpers defined.


In [ ]:
# ── Load function: Polars lazy scan with predicate pushdown ───────────────────
def load_split_lazy(
    parquet_files: list[Path],
    cols:          list[str],
    row_filter=None,
) -> pl.DataFrame:
    """
    Lazy-scan Parquet files with optional predicate pushdown, select columns, collect.
    """
    lf = pl.scan_parquet([str(f) for f in parquet_files], hive_partitioning=False)
    if row_filter is not None:
        lf = lf.filter(row_filter)
    available  = set(lf.collect_schema().names())
    cols_use   = [c for c in cols if c in available]
    missing    = [c for c in cols if c not in available]
    if missing:
        print(f"  ⚠  Columns not in schema (skipped): {missing}")
    return lf.select(cols_use).collect()

def hash_sample(lf: pl.LazyFrame, frac: float, seed: int = 0) -> pl.LazyFrame:
    """Deterministic loan-level hash sample from a Polars lazy frame."""
    n_buck = 1000
    n_keep = int(frac * n_buck)
    return lf.filter(
        pl.col("loan_sequence_number").cast(pl.Utf8).hash(seed=seed) % n_buck < n_keep
    )

print("✓  load_split_lazy() and hash_sample() defined.")

✓  load_split_lazy() and hash_sample() defined.


In [ ]:
# ── Load fitting training rows (obs_year < 2004) ──────────────────────
# These are the rows the final model is actually fitted on.
# The manifest train split covers 1999-2004, but obs_year == 2004 is reserved
# for early-stopping validation. The effective model training window is 1999-2003.

print("Loading training rows (obs_year < 2004, 1999-2003) ...")
t0 = time.time()

df_train = load_split_lazy(
    parquet_files=train_files,
    cols=TRAIN_LOAD_COLS,
    row_filter=pl.col("obs_year") < 2004,
)

elapsed = time.time() - t0

TRAIN_FIT_N_ROWS  = len(df_train)
TRAIN_FIT_POS     = int(df_train["y"].sum())
TRAIN_FIT_NEG     = TRAIN_FIT_N_ROWS - TRAIN_FIT_POS
TRAIN_FIT_POS_PCT = float(df_train["y"].mean() * 100)

print(f"  Rows loaded      : {TRAIN_FIT_N_ROWS:,}")
print(f"  Estimated size   : {df_train.estimated_size('gb'):.1f} GB")
print(f"  Elapsed          : {elapsed:.0f}s")
print(f"  Positive rate    : {TRAIN_FIT_POS_PCT:.4f}%  (1999-2003)")

# ── Schema contract checks inherited from NB03 ────────────────────────────────
valid_int_dtypes = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
}

for col in ["vintage_year", "vintage_quarter", "number_of_units"]:
    if col in df_train.columns:
        assert df_train.schema[col] in valid_int_dtypes, (
            f"{col} must be integer-coded before LightGBM training. "
            f"Found dtype={df_train.schema[col]!r}"
        )

print("\n✓  Integer categorical schema contract confirmed for training load.")

if "first_time_homebuyer_flag" in df_train.columns:
    fthb_null = int(df_train["first_time_homebuyer_flag"].is_null().sum())
    fthb_9    = int(
        df_train.select(
            pl.col("first_time_homebuyer_flag").cast(pl.Utf8).fill_null("__NULL__").eq("9").sum()
        ).item()
    )
    print(
        f"   first_time_homebuyer_flag raw distribution check: "
        f"NULL={fthb_null:,}, explicit '9'={fthb_9:,}"
    )

print("\nFitting categorical encoders on the training subset ...")
fit_cat_encoders(df_train)
df_train = encode_cats(df_train)

Loading training rows (obs_year < 2004, 1999–2003) ...
  Rows loaded      : 166,337,801
  Estimated size   : 26.4 GB
  Elapsed          : 1s
  Positive rate    : 1.2259%  (1999–2003)

✓  Integer categorical schema contract confirmed for training load.
   first_time_homebuyer_flag raw distribution check: NULL=0, explicit '9'=0

Fitting categorical encoders on the training subset ...
   Note: first_time_homebuyer_flag fitted without __NULL__ and 9. This is acceptable if those categories are absent from the obs_year < 2004 fitting window; later unseen values will be mapped to the reserved unknown category.
✓  LabelEncoders fitted:
   loan_purpose                 classes=  4  unknown_code=  4  has___NULL__=False  has_'9'=False
   occupancy_status             classes=  3  unknown_code=  3  has___NULL__=False  has_'9'=False
   property_type                classes=  6  unknown_code=  6  has___NULL__=False  has_'9'=False
   first_time_homebuyer_flag    classes=  3  unknown_code=  3  has___NULL

In [ ]:
# ── Load validation rows (obs_year == 2004) ───────────────────────────
# Uses the same LabelEncoders fitted on the training subset.

print("Loading validation rows (obs_year == 2004) ...")
t0 = time.time()

df_val = load_split_lazy(
    parquet_files=train_files,
    cols=TRAIN_LOAD_COLS,
    row_filter=pl.col("obs_year") == 2004,
)
df_val = encode_cats(df_val)

elapsed = time.time() - t0

VAL_N_ROWS    = len(df_val)
VAL_POS       = int(df_val["y"].sum())
VAL_POS_PCT   = float(df_val["y"].mean() * 100)

print(f"  Rows loaded      : {VAL_N_ROWS:,}")
print(f"  Estimated size   : {df_val.estimated_size('gb'):.1f} GB")
print(f"  Elapsed          : {elapsed:.0f}s")
print(f"  Positive rate    : {VAL_POS_PCT:.4f}%  (2004)")

combined_gb = df_train.estimated_size("gb") + df_val.estimated_size("gb")
print(f"\nCombined in-memory footprint: {combined_gb:.1f} GB")

Loading validation rows (obs_year == 2004) ...
  Rows loaded      : 81,753,843
  Estimated size   : 13.2 GB
  Elapsed          : 114s
  Positive rate    : 1.0136%  (2004)

Combined in-memory footprint: 39.9 GB


---
## Section 2 · Class Imbalance and Dataset Construction

### scale_pos_weight from the Actual Fitted Training Subset

`scale_pos_weight = n_negative / n_positive` is computed from the rows actually passed to
LightGBM for model fitting, i.e. the **1999-2003 fitted training subset** (`obs_year < 2004`),
not from the broader manifest `train` split. `obs_year == 2004` is reserved for early stopping
and model selection, so computing the ratio from the full manifest train split would leak
validation-period information into a training-objective quantity (Tashman, 2000; Kaufman et
al., 2012).

The ratio is a **practical inverse-frequency weighting convention** that makes the rare
positive class matter in the weighted binary objective (He & Garcia, 2009); it is not a formal
optimality result. Raw probabilities under this weighting are class-weighted ranking scores
rather than calibrated probabilities, and are recalibrated downstream before APS-style
probability-mass interpretation (see Key Design Decisions).

Undersampling is not used because it would change the row population and empirical class prior
seen by the learner; that would require explicit probability correction or recalibration
before using probability-mass-based scores downstream.

In [ ]:
# ── scale_pos_weight from actual fitted training rows ─────────────────────────
manifest_spw_est = MANIFEST.get("scale_pos_weight_estimate")

N_TRAIN       = TRAIN_FIT_N_ROWS
N_POS         = TRAIN_FIT_POS
N_NEG         = TRAIN_FIT_NEG
SCALE_POS_WGT = N_NEG / N_POS

print("Actual fitted training-subset statistics (obs_year < 2004):")
print(f"  Total rows         : {N_TRAIN:>12,}")
print(f"  Positive (y=1)     : {N_POS:>12,}  ({N_POS / N_TRAIN * 100:.4f}%)")
print(f"  Negative (y=0)     : {N_NEG:>12,}  ({N_NEG / N_TRAIN * 100:.4f}%)")
print(f"  scale_pos_weight   : {SCALE_POS_WGT:.4f}")

if manifest_spw_est is not None:
    print(f"\nManifest estimate (full train split 1999-2004): {manifest_spw_est:.4f}")
    print(f"Difference (fitted - manifest)                : {SCALE_POS_WGT - float(manifest_spw_est):+.4f}")
    print("A difference is expected: manifest covers 1999-2004; model is fitted on 1999-2003.")

Actual fitted training-subset statistics (obs_year < 2004):
  Total rows         :  166,337,801
  Positive (y=1)     :    2,039,217  (1.2259%)
  Negative (y=0)     :  164,298,584  (98.7741%)
  scale_pos_weight   : 80.5694

Manifest estimate (full train split 1999-2004): 85.5100
Difference (fitted - manifest)                : -4.9406
A difference is expected: manifest covers 1999-2004; model is fitted on 1999-2003.


In [ ]:
def df_to_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Extract (X float32, y int32) from a Polars DataFrame."""
    X = df.select(FEATURE_COLS).to_numpy(allow_copy=True).astype(np.float32)
    y = df["y"].to_numpy().astype(np.int32)
    return X, y

# df_train/df_val are only needed for encoder fitting, schema checks, and class-balance
# statistics. They are freed here; the LightGBM Datasets are built once in Section 4.
del df_train, df_val
gc.collect()

0

---
## Section 3 · Hyperparameter Search

### Optuna TPE on a Deterministic Loan-Level Sample

Hyperparameters are tuned on a deterministic `OPTUNA_SAMPLE_FRAC` loan-level sample drawn
from the same local training parquet files used for final fitting. Sampling at the loan level
avoids fragmenting repeated loan-month records across included and excluded samples.

The sample is split exactly like the full data - **sample fit-train** (`obs_year < 2004`) and
**sample early-stop validation** (`obs_year == 2004`) - preserving the same observation-time split as the full workflow (Tashman, 2000; Kaufman et al., 2012).

**Trial budget:**  
`N_OPTUNA_TRIALS` trials with Optuna's TPE sampler (Bergstra et al., 2011; Akiba et al., 2019)
and a MedianPruner (Optuna Developers, n.d.; see Key Design Decisions, §0). This is a practical
compute budget, not a guarantee that the global optimum is found. The configured behaviour is
`MedianPruner(n_startup_trials=15, n_warmup_steps=75, interval_steps=10)`, and each trial runs
up to 600 boosting rounds with early stopping at 40 rounds.

**`min_child_samples` scaling rule:**  
The Optuna-tuned value is multiplied by the integer scale factor
`int(N_TRAIN / sample_train_rows)` and the result is floored at 5000.
The scale factor uses fitted training rows only. The sampled 2004
validation rows must **not** enter the denominator - they are reserved for early stopping and
score comparison, and including them would systematically underscale `min_child_samples`. The
floor at 5000 is a conservative convention to limit leaf fragmentation at full-data scale, not
a formal optimality result.

In [ ]:
N_OPTUNA_THREADS   = 40
OPTUNA_SAMPLE_FRAC = 0.03
OPTUNA_SAMPLE_SEED = 42
N_OPTUNA_TRIALS    = 80

print("Optuna settings:")
print(f"  Threads     : {N_OPTUNA_THREADS}")
print(f"  Sample frac : {OPTUNA_SAMPLE_FRAC * 100:.0f}%")
print(f"  Trials      : {N_OPTUNA_TRIALS}")

print(f"Loading {OPTUNA_SAMPLE_FRAC * 100:.0f}% deterministic loan-level sample ...")
t0 = time.time()

lf_full = pl.scan_parquet([str(f) for f in train_files], hive_partitioning=False)

df_sample = (
    hash_sample(lf_full, OPTUNA_SAMPLE_FRAC, seed=OPTUNA_SAMPLE_SEED)
    .select(TRAIN_LOAD_COLS + ["loan_sequence_number"])
    .collect()
)
df_sample = encode_cats(df_sample)

df_smp_tr = df_sample.filter(pl.col("obs_year") < 2004)
df_smp_vl = df_sample.filter(pl.col("obs_year") == 2004)

sample_train_rows = len(df_smp_tr)
sample_val_rows   = len(df_smp_vl)
sample_train_pos  = int(df_smp_tr["y"].sum())
sample_val_pos    = int(df_smp_vl["y"].sum())

X_str, y_str = df_to_arrays(df_smp_tr)
X_svl, y_svl = df_to_arrays(df_smp_vl)

lgb_smp_tr = lgb.Dataset(X_str, label=y_str,
    feature_name=FEATURE_COLS, categorical_feature=cat_feature_names, free_raw_data=True)
lgb_smp_vl = lgb.Dataset(X_svl, label=y_svl,
    reference=lgb_smp_tr,
    feature_name=FEATURE_COLS, categorical_feature=cat_feature_names, free_raw_data=True)

del df_sample, df_smp_tr, df_smp_vl, X_str, y_str, X_svl, y_svl
gc.collect()

print(f"  Sample train : {sample_train_rows:,} rows  pos={sample_train_pos/sample_train_rows*100:.4f}%")
print(f"  Sample val   : {sample_val_rows:,} rows  pos={sample_val_pos/sample_val_rows*100:.4f}%")

Optuna settings:
  Threads     : 40
  Sample frac : 3%
  Trials      : 80
Loading 3% deterministic loan-level sample ...
  Sample train : 4,998,381 rows  pos=1.2269%
  Sample val   : 2,452,577 rows  pos=1.0144%


In [ ]:
from optuna_integration.lightgbm import LightGBMPruningCallback

def optuna_objective(trial: optuna.Trial) -> float:
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "device": "cpu",
        "force_row_wise": True,
        "num_threads": N_OPTUNA_THREADS,
        "seed": 42,
        "scale_pos_weight": SCALE_POS_WGT,
        "feature_pre_filter": False,

        # Optuna-search max_bin setting for the sampled data.
        "max_bin": 127,

        # Tree capacity: num_leaves is the primary complexity control in LightGBM's leaf-wise growth.
        # max_depth is searched as a secondary depth cap. We additionally constrain the search to
        # num_leaves <= 2^max_depth; this is a tuning convention, not a LightGBM requirement.
        "max_depth": trial.suggest_int("max_depth", 6, 9),
        "num_leaves": trial.suggest_int(
            "num_leaves", 63, min(255, 2 ** trial.params["max_depth"]), log=True
        ),

        # Learning-rate / boosting-round trade-off:
        "learning_rate": trial.suggest_float("learning_rate", 0.045, 0.09, log=True),

        # Row/feature subsampling:
        "feature_fraction": trial.suggest_float("feature_fraction", 0.65, 0.95),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.65, 0.95),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 3),

        # Leaf-size regularization on the sampled fitted-training rows.
        "min_child_samples": trial.suggest_int("min_child_samples", 750, 3000, log=True),

        # Split and weight regularization.
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 5.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-4, 25.0, log=True),
    }

    booster = lgb.train(
        params,
        lgb_smp_tr,
        num_boost_round=600,
        valid_sets=[lgb_smp_vl],
        valid_names=["valid_0"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=40,
                first_metric_only=True,
                verbose=False,
            ),
            lgb.log_evaluation(period=0),
            LightGBMPruningCallback(trial, "auc", "valid_0"),
        ],
    )

    return booster.best_score["valid_0"]["auc"]

print(f"Running {N_OPTUNA_TRIALS} Optuna trials ...")
t0 = time.time()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        multivariate=True,
        group=True,
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=15,
        n_warmup_steps=75,
        interval_steps=10,
    ),
)

study.optimize(
    optuna_objective,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

elapsed = time.time() - t0

print(f"\nOptuna complete in {elapsed / 60:.1f} min")
print(f"  Best AUROC  : {study.best_value:.5f}")
print("  Best params :")
for k, v in sorted(study.best_params.items()):
    print(f"    {k:<20}: {v}")

del lgb_smp_tr, lgb_smp_vl
gc.collect()

Running 80 Optuna trials ...


  0%|          | 0/80 [00:00<?, ?it/s]


Optuna complete in 46.7 min
  Best AUROC  : 0.93180
  Best params :
    bagging_fraction    : 0.6560465152415683
    bagging_freq        : 1
    feature_fraction    : 0.7687675193399143
    lambda_l1           : 0.007502171383644887
    lambda_l2           : 0.8507171183922538
    learning_rate       : 0.053643523941210136
    max_depth           : 9
    min_child_samples   : 2079
    min_gain_to_split   : 0.5226682849242004
    num_leaves          : 128


17

---
## Section 4 · Final Model Training

### Final Training and Artefact Save

The final model uses the best Optuna configuration on the full fitted training set. The model
family is LightGBM, an efficient implementation of gradient-boosted decision trees (Friedman,
2001; Ke et al., 2017).

`num_boost_round` is set to 1000 with early stopping patience of 25 rounds. This gives the
model room to converge while the chronological 2004 validation set selects the best iteration
before the ceiling binds. Early stopping is a model-selection control here, not a theoretical
optimality guarantee.

`min_child_samples` is multiplied by the integer scale factor `int(N_TRAIN / sample_train_rows)` (fitted rows only) and floored at 5000; the rationale and the
validation-row exclusion are detailed in §3. Raw predicted probabilities under
`scale_pos_weight` are not assumed to be calibrated - diagnostics are reported below and
isotonic recalibration is applied downstream before APS-style analysis (see Key Design
Decisions, §0).

After training, the following artefacts are saved:

- `lgbm_model.txt` - fitted LightGBM booster
- `cat_encoders.json` - categorical encoding contract for downstream prediction

In [ ]:
# Scale min_child_samples by integer factor int(N_TRAIN / sample_train_rows)
# (fitted rows only; 2004 validation rows excluded). Rationale in Section 3.

assert sample_train_rows > 0, "sample_train_rows must be > 0 for min_child_samples scaling."

sample_total_rows = sample_train_rows + sample_val_rows
SCALE_FACTOR      = max(1, int(N_TRAIN / sample_train_rows))

best_params      = dict(study.best_params)
optuna_min_child = int(best_params["min_child_samples"])
FINAL_MIN_CHILD  = max(int(optuna_min_child * SCALE_FACTOR), 5000)
best_params["min_child_samples"] = FINAL_MIN_CHILD

FINAL_PARAMS = {
    "objective"          : "binary",
    "metric"             : "auc",
    "boosting_type"      : "gbdt",
    "verbosity"          : -1,
    "device"             : "cpu",
    "num_threads"        : N_OPTUNA_THREADS,
    "force_row_wise"     : True,
    "seed"               : 42,
    "scale_pos_weight"   : SCALE_POS_WGT,
    # Practical final-training setting; not claimed as a universal CPU optimum.
    "max_bin"            : 255,
    "feature_pre_filter" : False,
    **best_params,
}

print("Final training hyperparameters:")
for k, v in sorted(FINAL_PARAMS.items()):
    print(f"  {k:<25}: {v}")

print("\nRow-count basis for scaling:")
print(f"  Full fitted training rows     : {N_TRAIN:,}")
print(f"  Sample fitted training rows   : {sample_train_rows:,}")
print(f"  Sample validation rows (2004) : {sample_val_rows:,}")
print(f"  Sample total rows             : {sample_total_rows:,}")

print(f"\nscale_pos_weight                : {SCALE_POS_WGT:.4f}")
print(f"min_child_samples scaling basis : {N_TRAIN:,} / {sample_train_rows:,} = {SCALE_FACTOR}")
print(f"min_child_samples final         : {optuna_min_child} × {SCALE_FACTOR} → {FINAL_MIN_CHILD}")

Final training hyperparameters:
  bagging_fraction         : 0.6560465152415683
  bagging_freq             : 1
  boosting_type            : gbdt
  device                   : cpu
  feature_fraction         : 0.7687675193399143
  feature_pre_filter       : False
  force_row_wise           : True
  lambda_l1                : 0.007502171383644887
  lambda_l2                : 0.8507171183922538
  learning_rate            : 0.053643523941210136
  max_bin                  : 255
  max_depth                : 9
  metric                   : auc
  min_child_samples        : 68607
  min_gain_to_split        : 0.5226682849242004
  num_leaves               : 128
  num_threads              : 40
  objective                : binary
  scale_pos_weight         : 80.56944601776074
  seed                     : 42
  verbosity                : -1

Row-count basis for scaling:
  Full fitted training rows     : 166,337,801
  Sample fitted training rows   : 4,998,381
  Sample validation rows (2004) : 2,452,577
 

In [ ]:
print("Reconstructing LightGBM Datasets for final training...")

# Remove any stale full-size objects from previous partial runs.
# No deletion of CAT_ENCODERS, UNKNOWN_CAT_CODES, FINAL_PARAMS, FEATURE_COLS, or cat_feature_names.
for obj in [
    "df_train", "df_val",
    "X_train", "y_train", "X_val", "y_val",
    "lgb_train", "lgb_val",
    "lgb_smp_tr", "lgb_smp_vl",
]:
    if obj in globals():
        del globals()[obj]

gc.collect()


# ── Rebuild fitted training Dataset ───────────────────────────────────────────
print("\nLoading and encoding fitted training rows (obs_year < 2004) ...")
t0 = time.time()

df_train = load_split_lazy(
    train_files,
    TRAIN_LOAD_COLS,
    pl.col("obs_year") < 2004,
)
df_train = encode_cats(df_train)

print(f"  df_train rows        : {len(df_train):,}")
print(f"  df_train size        : {df_train.estimated_size('gb'):.1f} GB")

X_train, y_train = df_to_arrays(df_train)
print(f"  X_train shape        : {X_train.shape}")
print(f"  X_train size         : {X_train.nbytes / 1e9:.1f} GB")

del df_train
gc.collect()

lgb_train = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=FEATURE_COLS,
    categorical_feature=cat_feature_names,
    params={"feature_pre_filter": FINAL_PARAMS["feature_pre_filter"]},
    free_raw_data=True,
)

# Force LightGBM to construct/bin the training Dataset now.
# This allows the raw numpy matrix to be released before loading validation data.
print("  Constructing LightGBM training Dataset ...")
lgb_train.construct()

del X_train, y_train
gc.collect()

print(f"  Training Dataset ready in {(time.time() - t0) / 60:.1f} min")


# ── Rebuild validation Dataset ────────────────────────────────────────────────
print("\nLoading and encoding validation rows (obs_year == 2004) ...")
t0 = time.time()

df_val = load_split_lazy(
    train_files,
    TRAIN_LOAD_COLS,
    pl.col("obs_year") == 2004,
)
df_val = encode_cats(df_val)

print(f"  df_val rows          : {len(df_val):,}")
print(f"  df_val size          : {df_val.estimated_size('gb'):.1f} GB")

X_val, y_val = df_to_arrays(df_val)
print(f"  X_val shape          : {X_val.shape}")
print(f"  X_val size           : {X_val.nbytes / 1e9:.1f} GB")

del df_val
gc.collect()

lgb_val = lgb.Dataset(
    X_val,
    label=y_val,
    reference=lgb_train,
    feature_name=FEATURE_COLS,
    categorical_feature=cat_feature_names,
    params={"feature_pre_filter": FINAL_PARAMS["feature_pre_filter"]},
    free_raw_data=True,
)

print("  Constructing LightGBM validation Dataset ...")
lgb_val.construct()

del X_val, y_val
gc.collect()

print(f"  Validation Dataset ready in {(time.time() - t0) / 60:.1f} min")


# ── Final training ────────────────────────────────────────────────────────────
print("\nTraining final model ...")
t0 = time.time()

BOOSTER = lgb.train(
    FINAL_PARAMS,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    valid_names=["val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=25, first_metric_only=True, verbose=True),
        lgb.log_evaluation(period=25),
    ],
)

elapsed      = time.time() - t0
best_iter    = BOOSTER.best_iteration
best_val_auc = BOOSTER.best_score["val"]["auc"]

print(f"\n✓  Training complete in {elapsed / 60:.1f} min")
print(f"   Best iteration : {best_iter}")
print(f"   Best val AUROC : {best_val_auc:.5f}")
print(f"   Trees          : {BOOSTER.num_trees()}")

del lgb_train, lgb_val
gc.collect()

Reconstructing LightGBM Datasets for final training...

Loading and encoding fitted training rows (obs_year < 2004) ...
  df_train rows        : 166,337,801
  df_train size        : 26.8 GB
  X_train shape        : (166337801, 29)
  X_train size         : 19.3 GB
  Constructing LightGBM training Dataset ...
  Training Dataset ready in 4.1 min

Loading and encoding validation rows (obs_year == 2004) ...
  df_val rows          : 81,753,843
  df_val size          : 13.2 GB
  X_val shape          : (81753843, 29)
  X_val size           : 9.5 GB
  Constructing LightGBM validation Dataset ...
  Validation Dataset ready in 2.0 min

Training final model ...
Training until validation scores don't improve for 25 rounds
[25]	val's auc: 0.923199
[50]	val's auc: 0.924669
[75]	val's auc: 0.925233
[100]	val's auc: 0.925608
[125]	val's auc: 0.925824
[150]	val's auc: 0.925997
[175]	val's auc: 0.926169
[200]	val's auc: 0.926335
[225]	val's auc: 0.926455
[250]	val's auc: 0.92653
[275]	val's auc: 0.926621

28

In [ ]:
# ── Save artefacts to local NVMe, then sync to Drive ──────────────────────────
BOOSTER.save_model(str(LOCAL_MODELS_DIR / "lgbm_model.txt"))

# LabelEncoder classes
enc_data = {col: list(le.classes_.astype(str)) for col, le in CAT_ENCODERS.items()}
with open(LOCAL_MODELS_DIR / "cat_encoders.json", "w") as f:
    json.dump(enc_data, f, indent=2)

print("✓  Model artefacts written to local NVMe:")
for fname in ["lgbm_model.txt", "cat_encoders.json"]:
    p = LOCAL_MODELS_DIR / fname
    print(f"   {fname:<30}  {p.stat().st_size/1e6:.1f} MB")

# ── Sync to Drive ─────────────────────────────────────────────────────────────
print("\nSyncing to Drive ...")
for fname in ["lgbm_model.txt", "cat_encoders.json"]:
    shutil.copy2(LOCAL_MODELS_DIR / fname, MODELS_DIR / fname)
print(f"✓  Artefacts synced to: {MODELS_DIR}")

✓  Model artefacts written to local NVMe:
   lgbm_model.txt                  7.6 MB
   cat_encoders.json               0.0 MB

Syncing to Drive ...
✓  Artefacts synced to: /content/drive/MyDrive/master_thesis/models


---
## Section 5 · Model Evaluation

### Evaluation Design

Evaluates three properties relevant to downstream conformal prediction workflows:

1. **Discrimination across regimes** - AUROC (primary, threshold-free; Fawcett, 2006),
   Average Precision (AP; informative under class imbalance; Davis & Goadrich, 2006), and
   Brier score (Brier, 1950) on calibration and all test regimes. AP uses sklearn's
   step-function `average_precision_score`; cross-regime AP comparisons are prevalence-
   sensitive (see Key Design Decisions, §0).

2. **Local discrimination** - subgroup AUROC on the calibration split: whether the score is
   informative within subgroup cells. This is a diagnostic, not a Mondrian validity proof
   (detailed in §5.1).

3. **Probability calibration and feature attribution** - bin-based calibration diagnostics,
   Guo-style equal-width ECE, equal-frequency calibration summaries, and SHAP. SHAP describes
   additive model attribution for the fitted tree ensemble (Lundberg & Lee, 2017; Lundberg
   et al., 2019); the non-causal reading is this notebook's modeling caveat, not a theorem
   from those papers.

The model is deliberately evaluated across temporal regimes to document how a score function
trained only on the early-history window behaves under later distributional shift. This is an
out-of-time transport diagnostic, not a claim that the early model is universally stable.

In [ ]:
def df_to_X(df: pl.DataFrame) -> np.ndarray:
    """Extract model feature matrix from a Polars DataFrame."""
    return df.select(FEATURE_COLS).to_numpy(allow_copy=True).astype(np.float32)

EVAL_FRAC        = 0.02
RELIABILITY_FRAC = 0.01
SHAP_FRAC        = 0.001
EVAL_SPLITS = ["calibration","test_subprime","test_normal","test_covid","test_rate_hike"]

print(f"Evaluation: {EVAL_FRAC*100:.0f}% samples across {len(EVAL_SPLITS)} splits.")

Evaluation: 2% samples across 5 splits.


### 5.1 · Per-Split AUROC / AP / Brier and Subgroup AUROC

This section evaluates the fitted LightGBM model across the downstream temporal regimes defined in NB03.

In [ ]:
MIN_SUBGROUP_TOTAL = 5_000
MIN_SUBGROUP_POS   = 100
MIN_SUBGROUP_NEG   = 100

def eligible_binary_subgroup(mask: np.ndarray, y: np.ndarray) -> bool:
    """Check whether a subgroup has sufficient size and class support for AUROC."""
    n = int(mask.sum())
    if n < MIN_SUBGROUP_TOTAL:
        return False
    n_pos = int(y[mask].sum())
    return (n_pos >= MIN_SUBGROUP_POS) and ((n - n_pos) >= MIN_SUBGROUP_NEG)

eval_results: dict = {}

for split_name in EVAL_SPLITS:
    t0 = time.time()

    split_dir = PANEL_DIR / f"split={split_name}"
    files = sorted(split_dir.glob("*.parquet"))

    lf     = pl.scan_parquet([str(f) for f in files], hive_partitioning=False)
    avail  = set(lf.collect_schema().names())
    cols_load = [c for c in EVAL_LOAD_COLS if c in avail]

    df     = hash_sample(lf, EVAL_FRAC, seed=99).select(cols_load).collect()
    subgroup_label_arrays = {
        c: df[c].to_numpy().astype(str)
        for c in SUBGROUP_COLS if c in df.columns
    }

    df     = encode_cats(df)

    y      = df["y"].to_numpy().astype(np.int8)
    p_hat  = BOOSTER.predict(df_to_X(df), num_threads=-1).astype(np.float32)

    auroc  = float(roc_auc_score(y, p_hat))
    ap     = float(average_precision_score(y, p_hat))
    brier  = float(brier_score_loss(y, p_hat))

    base_rate      = float(y.mean())
    brier_baseline = base_rate * (1.0 - base_rate)
    bss            = 1.0 - brier / brier_baseline if brier_baseline > 0 else float("nan")

    res = {
        "auroc": auroc, "ap": ap, "brier": brier,
        "brier_baseline": brier_baseline, "brier_skill_score": bss,
        "n_rows": len(df), "pos_rate_pct": float(y.mean() * 100),
    }

    # ── Subgroup AUROC on calibration split ──────────────────────────────────
    # Diagnostic only: subgroup AUROC measures local discrimination. It is not
    # itself a Mondrian validity guarantee; group-conditional conformal validity
    # depends on within-group calibration/exchangeability and quantile resolution.
    if split_name == "calibration":
        sg_auroc      = {}
        sg_eligibility = {}
        for sg_col in SUBGROUP_COLS:
            if sg_col not in df.columns:
                continue
            sg_arr     = subgroup_label_arrays[sg_col]
            aucs       = {}
            eligibility = {}
            for g in np.unique(sg_arr):
                mask  = sg_arr == g
                n_pos = int(y[mask].sum())
                n_neg = int(mask.sum()) - n_pos
                elig  = eligible_binary_subgroup(mask, y)
                eligibility[g] = {"n": int(mask.sum()), "n_pos": n_pos, "n_neg": n_neg, "eligible": elig}
                if elig:
                    aucs[g] = float(roc_auc_score(y[mask], p_hat[mask]))
            sg_auroc[sg_col]       = aucs
            sg_eligibility[sg_col] = eligibility
        res["subgroup_auroc"]       = sg_auroc
        res["subgroup_eligibility"] = sg_eligibility

    # ── COVID workout-plan subgroup analysis ──────────────────────────────────
    # in_workout_plan_flag captures F (Forbearance) + R (Repayment) + T (Trial).
    # It is metadata only, never an active model feature.
    # Labelled "in_workout_plan" throughout, not "in_forbearance".
    if split_name == "test_covid" and "in_workout_plan_flag" in df.columns:
        wp_series = df["in_workout_plan_flag"]
        assert wp_series.dtype in (
            pl.Int8, pl.Int16, pl.Int32, pl.Int64,
            pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        ), (
            f"in_workout_plan_flag must be integer-encoded (0/1) for subgroup "
            f"analysis. Got dtype={wp_series.dtype!r}. Check NB03 panel construction."
        )
        wp = wp_series.to_numpy()
        workout_aucs = {}
        for grp, label in [(0, "not_in_workout_plan"), (1, "in_workout_plan")]:
            mask = wp == grp
            if eligible_binary_subgroup(mask, y):
                workout_aucs[label] = float(roc_auc_score(y[mask], p_hat[mask]))
        res["workout_plan_auroc"] = workout_aucs

    eval_results[split_name] = res

    extra = ""
    if split_name == "test_covid" and res.get("workout_plan_auroc"):
        wp_str = " | ".join(f"{k}={v:.4f}" for k, v in res["workout_plan_auroc"].items())
        extra = f"  | workout_plan: [{wp_str}]"

    print(
        f"  {split_name:<18}: AUROC={auroc:.4f}  AP={ap:.4f}  "
        f"Brier={brier:.4f}  Brier_base={brier_baseline:.4f}  BSS={bss:+.3f}  "
        f"n={len(df):,}  ({time.time()-t0:.0f}s){extra}"
    )

    del df, y, p_hat
    gc.collect()

neg_bss_splits = [s for s in EVAL_SPLITS if eval_results[s]["brier_skill_score"] < 0]
if neg_bss_splits:
    print(f"\n⚠  Raw-score Brier does not beat the constant base-rate baseline on: "
          f"{', '.join(neg_bss_splits)}")
    print("   The model is trained with scale_pos_weight, so raw outputs are class-weighted")
    print("   ranking scores, not calibrated probability estimates. Brier/ECE in this notebook")
    print("   characterise the RAW scores only; calibrated probabilities are produced by the")
    print("   isotonic recalibration in NB05a.")
else:
    print("\n✓  Raw-score Brier beats the constant base-rate baseline on all evaluated splits.")

print("\n✓  Evaluation complete.")

  calibration       : AUROC=0.9140  AP=0.3361  Brier=0.0640  Brier_base=0.0111  BSS=-4.750  n=3,592,673  (269s)
  test_subprime     : AUROC=0.8833  AP=0.3494  Brier=0.0605  Brier_base=0.0246  BSS=-1.462  n=12,669,220  (699s)
  test_normal       : AUROC=0.8723  AP=0.3135  Brier=0.0403  Brier_base=0.0151  BSS=-1.673  n=15,096,097  (1031s)
  test_covid        : AUROC=0.8262  AP=0.2114  Brier=0.0390  Brier_base=0.0163  BSS=-1.394  n=4,032,323  (176s)  | workout_plan: [not_in_workout_plan=0.7876 | in_workout_plan=0.6429]
  test_rate_hike    : AUROC=0.8719  AP=0.2508  Brier=0.0289  Brier_base=0.0101  BSS=-1.851  n=6,127,905  (502s)

⚠  Raw-score Brier does not beat the constant base-rate baseline on: calibration, test_subprime, test_normal, test_covid, test_rate_hike
   The model is trained with scale_pos_weight, so raw outputs are class-weighted
   ranking scores, not calibrated probability estimates. Brier/ECE in this notebook
   characterise the RAW scores only; calibrated probabilities a

### 5.2 · Evaluation Tables

In [ ]:
from IPython.display import display

SPLIT_LABELS = {
    "calibration"    : "Calibration (2005-06)",
    "test_subprime"  : "Subprime (2007-12)",
    "test_normal"    : "Normal (2013-Sep 2019)",
    "test_covid"     : "COVID (2020-21)",
    "test_rate_hike" : "Rate Hike (2022-23)",
}

# Structured evaluation table
table_rows = []
for s in EVAL_SPLITS:
    table_rows.append({
        "Regime": SPLIT_LABELS.get(s, s),
        "AUROC (↑)": eval_results[s]["auroc"],
        "AP (↑)": eval_results[s]["ap"],
        "Brier Score (↓)": eval_results[s]["brier"]
    })

df_metrics = pd.DataFrame(table_rows).set_index("Regime")

#  Title and Table
print(f"LightGBM Discrimination Across Temporal Regimes")
print(f"Target: {TARGET_LABEL} (12-month forward, transition-risk setup)")
print("-" * 80)
display(df_metrics.style.format("{:.3f}").set_caption("Model Performance Split Metrics"))
print("-" * 80)

# Diagnostic & Baseline
calib_auroc = eval_results["calibration"]["auroc"]
print("\n=== AUROC vs. Calibration baseline ===")
for s in EVAL_SPLITS:
    delta = eval_results[s]["auroc"] - calib_auroc
    flag = "Δ" if abs(delta) > 0.05 else "."
    print(f" {flag} {s:<18}: {eval_results[s]['auroc']:.4f}  (Δ {delta:+.4f} vs calibration)")

print("\nRegime caveats (test_normal Q4-2019 trim, COVID policy effects, rate-hike horizon)")
print("truncation are documented in §5.1 and the NB04 manifest notes.")

rh_rate = eval_results.get("test_rate_hike", {}).get("pos_rate_pct")
nm_rate = eval_results.get("test_normal", {}).get("pos_rate_pct")
if rh_rate is not None and nm_rate is not None and rh_rate < nm_rate:
    print(f"\nNote: test_rate_hike positive rate ({rh_rate:.4f}%) is below test_normal ")
    print(f"      ({nm_rate:.4f}%) here; the 12-month label window for late-2023 closes mid-2024,")
    print(f"      before slow rate-stress delinquency fully materialises (label-design horizon)")
    print(f"      truncation; computed, not assumed.")

LightGBM Discrimination Across Temporal Regimes
Target: 60+ DPD or REO Acquisition (12-month forward, transition-risk setup)
--------------------------------------------------------------------------------


,AUROC (↑),AP (↑),Brier Score (↓)
Regime,,,
Calibration (2005-06),0.914,0.336,0.064
Subprime (2007-12),0.883,0.349,0.060
Normal (2013-Sep 2019),0.872,0.313,0.040
COVID (2020-21),0.826,0.211,0.039
Rate Hike (2022-23),0.872,0.251,0.029


--------------------------------------------------------------------------------

=== AUROC vs. Calibration baseline ===
 . calibration       : 0.9140  (Δ +0.0000 vs calibration)
 . test_subprime     : 0.8833  (Δ -0.0307 vs calibration)
 . test_normal       : 0.8723  (Δ -0.0417 vs calibration)
 Δ test_covid        : 0.8262  (Δ -0.0878 vs calibration)
 . test_rate_hike    : 0.8719  (Δ -0.0421 vs calibration)

Regime caveats (test_normal Q4-2019 trim, COVID policy effects, rate-hike horizon)
truncation are documented in §5.1 and the NB04 manifest notes.

Note: test_rate_hike positive rate (1.0234%) is below test_normal 
      (1.5317%) here; the 12-month label window for late-2023 closes mid-2024,
      before slow rate-stress delinquency fully materialises (label-design horizon)
      truncation; computed, not assumed.


> **Window note.** `test_subprime` is the legacy pipeline key for the thesis's
> **Crisis (2007–12)** window. The rate-hike window is not right-truncated:
> its last 2023 observation closes its 12-month label horizon in December 2024,
> within the available performance history. The limitation is the fixed
> 12-month horizon relative to potentially slower credit effects, not missing
> follow-up.

### 5.3 · Subgroup AUROC (Calibration Split)

In [ ]:
sg_aurocs = eval_results.get("calibration", {}).get("subgroup_auroc", {})

if sg_aurocs:
    print(f"Calibration-Split Subgroup AUROC  |  Target: {TARGET_LABEL}")
    print(f"Eligibility: n ≥ {MIN_SUBGROUP_TOTAL:,}, positives ≥ {MIN_SUBGROUP_POS}, negatives ≥ {MIN_SUBGROUP_NEG}")
    print("=" * 90)

    for sg_col, aucs in sg_aurocs.items():
        table_rows = []

        # Sort by AUROC value to maintain a clear ascending performance view
        for g, v in sorted(aucs.items(), key=lambda x: x[1]):
            # Map performance tier categories
            if v < 0.70:
                tier = "Underperforming (< 0.70)"
            elif v > 0.80:
                tier = "Strong (>= 0.80)"
            else:
                tier = "Acceptable"

            table_rows.append({
                "Subgroup Value": g[:28],
                "AUROC (↑)": v,
                "Performance Tier": tier
            })

        df_sg = pd.DataFrame(table_rows).set_index("Subgroup Value")

        # Format the title and print the table
        title = sg_col.replace("_", " ").title()
        print(f"Subgroup Analysis: {title}")
        print(df_sg)
        print("\n" + "-" * 90 + "\n")

Calibration-Split Subgroup AUROC  |  Target: 60+ DPD or REO Acquisition
Eligibility: n ≥ 5,000, positives ≥ 100, negatives ≥ 100
Subgroup Analysis: Fico Tier
                AUROC (↑)  Performance Tier
Subgroup Value                             
Subprime         0.851812  Strong (>= 0.80)
Prime            0.858644  Strong (>= 0.80)
Near-Prime       0.864761  Strong (>= 0.80)
Sentinel         0.914833  Strong (>= 0.80)

------------------------------------------------------------------------------------------

Subgroup Analysis: Ltv Bucket
                AUROC (↑)  Performance Tier
Subgroup Value                             
High             0.859780  Strong (>= 0.80)
Moderate         0.889095  Strong (>= 0.80)
Low              0.910357  Strong (>= 0.80)

------------------------------------------------------------------------------------------

Subgroup Analysis: Fico Ltv Group
                       AUROC (↑)  Performance Tier
Subgroup Value                                    
Prime 

### 5.4 · Calibration Error Diagnostic Tables

The calibration error diagnostic tables show whether predicted probabilities are calibrated in absolute terms: a mean predicted probability below the observed positive rate indicates under-prediction, above indicates over-prediction (Niculescu-Mizil & Caruana, 2005; Guo et al., 2017).

Two related bin-based calibration summaries are reported:

1. **Equal-width ECE:** `N_BINS` equal-width bins on `[0, 1]`, using the ECE definition of Guo et al. (2017) (weighted mean of |acc - conf|). Guo et al. use M = 15; M = `N_BINS` is used here.
2. **Equal-frequency calibration error (QECE):** `N_BINS` percentile bins. Useful because predicted probabilities are highly concentrated near one end of `[0, 1]`; it is not the same numerical quantity as the equal-width ECE.

Because the model uses `scale_pos_weight`, raw probabilities are not assumed calibrated and are corrected downstream before APS-style evaluation (see Key Design Decisions, §0). For conformal prediction, marginal coverage comes from held-out score calibration, not from perfect probability calibration of the base model; probability calibration still matters for interpretation and for APS, which uses cumulative probability mass rather than a threshold-free ranking score (Romano et al., 2020).

**Interpretation caveat:** these tables are computed on the calibration split;
late-2006 observations have 12-month label windows extending into 2007. They
diagnose the calibration population rather than a horizon-purged tranquil-period
sample.

In [ ]:
print("Loading 1% calibration sample for reliability assessment ...")

calib_dir   = PANEL_DIR / "split=calibration"
calib_files = sorted(calib_dir.glob("*.parquet"))

lf_r = pl.scan_parquet([str(f) for f in calib_files], hive_partitioning=False)
df_r = (
    hash_sample(lf_r, RELIABILITY_FRAC, seed=1)
    .select([c for c in EVAL_LOAD_COLS if c != "loan_sequence_number"])
    .collect()
)

df_r = encode_cats(df_r)

p_r = BOOSTER.predict(df_to_X(df_r), num_threads=-1).astype(np.float32)
y_r = df_r["y"].to_numpy().astype(np.int8)
del df_r
gc.collect()

print(f" {len(y_r):,} calibration rows")

N_BINS = 10

def bin_calibration_stats(
    p: np.ndarray,
    y: np.ndarray,
    edges: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """
    Compute bin-wise calibration statistics.

    For Guo-style ECE, pass equal-width edges on [0, 1].
    For equal-frequency/QECE diagnostics, pass percentile edges.

    Returns:
        bin_midpoints, mean_pred, frac_pos, counts, calibration_error
    """
    edges = edges.astype(np.float64).copy()
    edges[0]  = -np.inf
    edges[-1] = np.inf

    idx = np.clip(np.digitize(p, edges, right=True) - 1, 0, len(edges) - 2)

    bin_midpoints = []
    mean_pred     = []
    frac_pos      = []
    counts        = []

    for b in range(len(edges) - 1):
        mask = idx == b
        n_b = int(mask.sum())
        counts.append(n_b)

        if n_b == 0:
            bin_midpoints.append(np.nan)
            mean_pred.append(np.nan)
            frac_pos.append(np.nan)
        else:
            bin_midpoints.append(float(p[mask].mean()))
            mean_pred.append(float(p[mask].mean()))
            frac_pos.append(float(y[mask].mean()))

    bin_midpoints = np.array(bin_midpoints, dtype=float)
    mean_pred     = np.array(mean_pred, dtype=float)
    frac_pos      = np.array(frac_pos, dtype=float)
    counts        = np.array(counts, dtype=int)

    nonempty = counts > 0
    cal_err = float(
        np.average(
            np.abs(frac_pos[nonempty] - mean_pred[nonempty]),
            weights=counts[nonempty],
        )
    )

    return bin_midpoints, mean_pred, frac_pos, counts, cal_err


# Binning Computations

# Equal-width ECE (Guo et al. 2017 definition)
edges_width = np.linspace(0.0, 1.0, N_BINS + 1)
_, ew_mean_pred, ew_frac_pos, ew_counts, ece_equal_width_m10 = bin_calibration_stats(
    p_r, y_r, edges_width
)

# Equal-frequency diagnostic (Percentile bins)
edges_freq = np.percentile(p_r, np.linspace(0, 100, N_BINS + 1))
edges_freq = np.maximum.accumulate(edges_freq)
_, q_mean_pred, q_frac_pos, q_counts, qece_equal_frequency = bin_calibration_stats(
    p_r, y_r, edges_freq
)

ece  = ece_equal_width_m10
qece = qece_equal_frequency


# Tabular Display
print(f"\nCalibration Table Diagnostics | Target: {TARGET_LABEL}")
print("ECE = Guo-style equal-width; QECE = equal-frequency percentile diagnostic.")
print("=" * 90)

# Generate Equal-Width Table
df_ew = pd.DataFrame({
    "Bin Index": range(1, N_BINS + 1),
    "Mean Predicted Prob": ew_mean_pred,
    "Observed Positive Rate": ew_frac_pos,
    "Bin Sample Count": ew_counts
}).set_index("Bin Index")

display(df_ew.style.format({
    "Mean Predicted Prob": "{:.4f}",
    "Observed Positive Rate": "{:.4f}",
    "Bin Sample Count": "{:,}"
}).set_caption(f"Equal-Width Binning Strategy (ECE: {ece:.4f})"))

print("\n" + "-" * 90 + "\n")

# Generate Equal-Frequency Table
df_q = pd.DataFrame({
    "Bin Index": range(1, N_BINS + 1),
    "Mean Predicted Prob": q_mean_pred,
    "Observed Positive Rate": q_frac_pos,
    "Bin Sample Count": q_counts
}).set_index("Bin Index")

display(df_q.style.format({
    "Mean Predicted Prob": "{:.4f}",
    "Observed Positive Rate": "{:.4f}",
    "Bin Sample Count": "{:,}"
}).set_caption(f"Equal-Frequency Percentile Binning Strategy (QECE: {qece:.4f})"))

print("=" * 90)


# Logs & Diagnostics
print()
print(f"Equal-width ECE (M=10, Guo et al. 2017 definition) = {ece:.4f}")
print(f"Equal-frequency calibration error (QECE) = {qece:.4f}")

# Calibration context
base_rate_r = float(y_r.mean())
mean_pred_r = float(p_r.mean())
gap_r       = mean_pred_r - base_rate_r
rel_word    = "above" if gap_r > 0 else "below" if gap_r < 0 else "equal to"

print(f"\nObserved positive rate (1% calibration sample) : {base_rate_r:.4f}")
print(f"Mean raw predicted probability                  : {mean_pred_r:.4f}")
print(f"Mean prediction is {rel_word} the observed rate by {abs(gap_r):.4f}.")
print("Design context: training uses scale_pos_weight, so raw outputs are not calibrated")
print("probabilities by construction. The ECE/QECE values above quantify the raw-score")
print("calibration gap that the isotonic recalibration step in NB05a is designed to remove.")

del p_r, y_r
gc.collect()

Loading 1% calibration sample for reliability assessment ...
 1,797,233 calibration rows

Calibration Table Diagnostics | Target: 60+ DPD or REO Acquisition
ECE = Guo-style equal-width; QECE = equal-frequency percentile diagnostic.


,Mean Predicted Prob,Observed Positive Rate,Bin Sample Count
Bin Index,,,
1,0.0363,0.0013,"1,074,922"
2,0.1432,0.0036,"262,346"
3,0.2461,0.0070,"140,655"
4,0.3471,0.0107,"94,556"
5,0.4473,0.0161,"71,871"
6,0.5468,0.0192,"51,463"
7,0.6462,0.0282,"34,598"
8,0.7461,0.0467,"21,243"
9,0.8502,0.0931,"16,484"



------------------------------------------------------------------------------------------



,Mean Predicted Prob,Observed Positive Rate,Bin Sample Count
Bin Index,,,
1,0.0075,0.0006,"179,725"
2,0.0150,0.0008,"179,722"
3,0.0241,0.0010,"179,723"
4,0.0365,0.0012,"179,723"
5,0.0542,0.0021,"179,726"
6,0.0820,0.0023,"179,721"
7,0.1279,0.0033,"179,731"
8,0.2087,0.0059,"179,715"
9,0.3531,0.0107,"179,723"



Equal-width ECE (M=10, Guo et al. 2017 definition) = 0.1471
Equal-frequency calibration error (QECE) = 0.1471

Observed positive rate (1% calibration sample) : 0.0112
Mean raw predicted probability                  : 0.1583
Mean prediction is above the observed rate by 0.1471.
Design context: training uses scale_pos_weight, so raw outputs are not calibrated
probabilities by construction. The ECE/QECE values above quantify the raw-score
calibration gap that the isotonic recalibration step in NB05a is designed to remove.


66

### Why the raw scores require calibration

The model was fitted with `scale_pos_weight ≈ 80.57`, computed from the
1999–2003 fitting rows, whose positive rate is 1.23%. The resulting raw outputs
are not treated as calibrated event probabilities. The calibration diagnostics
confirm substantial overprediction, so NB05a subsequently fits isotonic
recalibration.

On the 1% calibration diagnostic sample, mean raw predicted probability is
0.1583 against an observed positive rate of 0.0112, with ECE = 0.1471. The
equal-width bins show that the distortion is not confined to the tail: the
lowest bin predicts 3.63% where the empirical rate is 0.13%, and the top bin
predicts 95.9% against an observed 34.7%.

This is a calibration-level problem, not a failure of discrimination: AUROC is
0.914 on calibration and 0.826–0.883 across the four test regimes.
Recalibration is not required for split-conformal marginal validity; NB05a uses
it because APS depends on the probability vector and the drift, operational,
and cross-model analyses need a calibrated probability scale. Isotonic
calibration is monotone non-decreasing but can create ties, so ranking is
preserved only up to ties.

### 5.5 · SHAP Feature Importance

TreeSHAP (`shap.TreeExplainer`) is computed on a 20,000-row sample drawn from the **calibration split** (obs_year 2005-2006). SHAP values provide additive feature-attribution summaries for the fitted model; TreeSHAP is the efficient tree-ensemble version of this framework (Lundberg & Lee, 2017; Lundberg et al., 2019). SHAP values are computed on the raw log-odds (margin) scale - the default `model_output='raw'` for `shap.TreeExplainer` with a binary LightGBM booster. The mean |SHAP value| used for feature ranking is therefore on the raw
log-odds scale and is not directly interpretable as a probability change. The
ranking reported below is the raw-scale ranking; its stability is assessed
empirically across two independent calibration samples.

Computing SHAP on out-of-time calibration data rather than fitted training data shows which features the trained score function relies on when applied outside its fitting window. This is directly relevant for interpreting score behavior in the CP regime-stability analysis.

**Near-zero SHAP features:**
Several features contribute near-zero mean absolute SHAP attribution on this calibration sample; the ranking is shown in the summary table below. They are retained in the active feature set because removing them would break the encoding/feature-order contract.

Their near-zero SHAP values should be interpreted narrowly: they indicate that the fitted model assigns negligible model-attribution mass to these variables on this evaluated sample. They do **not** prove causal irrelevance, structural irrelevance, or universal harmlessness. This non-causal interpretation caveat is the notebook's own modeling caveat; Lundberg and Lee (2017) and Lundberg et al. (2019) are cited for SHAP / TreeSHAP as model-attribution methods, not as sources for causal identification. The `ltv_missing` and `cltv_missing` flags in particular have essentially zero prevalence in the 1999-2006 window because the underlying fields were routinely disclosed in that era.

In [ ]:
SHAP_MAX_ROWS = 20_000

print(f"Computing SHAP values on up to {SHAP_MAX_ROWS:,} calibration rows ...")

lf_sh = pl.scan_parquet([str(f) for f in calib_files], hive_partitioning=False)
df_sh = hash_sample(lf_sh, SHAP_FRAC, seed=7).select(FEATURE_COLS).collect()
df_sh = encode_cats(df_sh)

if len(df_sh) > SHAP_MAX_ROWS:
    df_sh = df_sh.sample(SHAP_MAX_ROWS, seed=7)

X_sh = df_sh.to_numpy(allow_copy=True).astype(np.float32)
del df_sh
gc.collect()

print(f"  SHAP background rows: {X_sh.shape[0]:,}")

explainer = shap.TreeExplainer(BOOSTER)
shap_vals = explainer.shap_values(X_sh)
del X_sh
gc.collect()

mean_abs = np.abs(shap_vals).mean(axis=0)
shap_df = (
    pd.DataFrame({"feature": FEATURE_COLS, "mean_abs_shap": mean_abs})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

# Text-Based Tabular Display
print("\nSHAP Feature Importance Summary")
print("Values represent global mean absolute SHAP attribution on the raw log-odds scale.")
print("=" * 60)
print(shap_df.to_string(index=False, float_format="{:.4f}".format))
print("=" * 60)

# Downstream Stability and Dead-Feature Diagnostics
# SHAP ranking stability: independent second sample, different hash seed ---
df_sh2 = hash_sample(lf_sh, SHAP_FRAC, seed=13).select(FEATURE_COLS).collect()
df_sh2 = encode_cats(df_sh2)

if len(df_sh2) > SHAP_MAX_ROWS:
    df_sh2 = df_sh2.sample(SHAP_MAX_ROWS, seed=13)

X_sh2 = df_sh2.to_numpy(allow_copy=True).astype(np.float32)
del df_sh2
gc.collect()

shap_vals2 = explainer.shap_values(X_sh2)
del X_sh2
gc.collect()

mean_abs2 = np.abs(shap_vals2).mean(axis=0)

shap_rank_spearman = float(
    pd.Series(mean_abs, index=FEATURE_COLS)
    .corr(pd.Series(mean_abs2, index=FEATURE_COLS), method="spearman")
)

TOP_K = 10
top1 = set(pd.Series(mean_abs, index=FEATURE_COLS).nlargest(TOP_K).index)
top2 = set(pd.Series(mean_abs2, index=FEATURE_COLS).nlargest(TOP_K).index)
top_k_overlap = len(top1 & top2)

print("\n=== SHAP ranking stability (two independent calibration samples) ===")
print(f"  Spearman rank correlation (seed 7 vs seed 13) : {shap_rank_spearman:.4f}")
print(f"  Top-{TOP_K} feature overlap                          : {top_k_overlap}/{TOP_K}")

sym_diff = sorted(top1 ^ top2)
if sym_diff:
    print(f"  Features in only one top-{TOP_K} set             : {sym_diff}")

SHAP_RANK_STABILITY_MIN = 0.90
if shap_rank_spearman < SHAP_RANK_STABILITY_MIN:
    print(f"\nΔ  Rank correlation below {SHAP_RANK_STABILITY_MIN:.2f}: the SHAP ranking is sample-sensitive.")
    print("   Individual feature ranks (especially mid-table) should not be quoted as point findings;")
    print("   report only rank ranges that are stable across both samples.")
else:
    print(f"\n✓  SHAP ranking is stable across independent samples (ρ ≥ {SHAP_RANK_STABILITY_MIN:.2f}).")

# Dead-feature check: zero SHAP contribution
zero_shap_features = shap_df.loc[shap_df["mean_abs_shap"] <= 0.0, "feature"].tolist()

if zero_shap_features:
    split_counts = dict(zip(BOOSTER.feature_name(),
                            BOOSTER.feature_importance(importance_type="split")))
    print(f"\nΔ  Features with zero mean |SHAP| on the calibration sample ({len(zero_shap_features)}):")
    confirmed_dead = []
    for f in zero_shap_features:
        n_splits = int(split_counts.get(f, 0))
        print(f"     - {f:<32} booster split count = {n_splits}")
        if n_splits == 0:
            confirmed_dead.append(f)
else:
    print("\n✓  Every active feature has non-zero SHAP contribution on the calibration sample.")

Computing SHAP values on up to 20,000 calibration rows ...
  SHAP background rows: 20,000

SHAP Feature Importance Summary
Values represent global mean absolute SHAP attribution on the raw log-odds scale.
                    feature  mean_abs_shap
               credit_score         0.7665
          is_multi_borrower         0.3986
      current_interest_rate         0.3798
               original_ltv         0.3148
          upb_rel_change_3m         0.2508
      months_since_last_dlq         0.1896
               original_dti         0.1886
               original_upb         0.1676
     original_interest_rate         0.1106
               loan_purpose         0.1037
            census_division         0.0840
              original_cltv         0.0711
         original_loan_term         0.0665
              is_30dpd_at_t         0.0634
               vintage_year         0.0633
              property_type         0.0560
                current_upb         0.0500
            vintage_q

---
## Section 6 · Artefact Registry

`model_manifest.json` is the machine-readable contract for downstream notebooks. It records:

- the active model feature list used by the fitted model (cross-checked against `panel_manifest.json`)
- `scale_pos_weight`, the model's best iteration, and best validation AUROC
- per-split sample-based evaluation metrics (AUROC / AP / Brier) across all regimes


In [ ]:
per_split_metrics = {
    s: {
        "auroc"             : round(eval_results[s]["auroc"], 6),
        "ap"                : round(eval_results[s]["ap"], 6),
        "brier"             : round(eval_results[s]["brier"], 6),
        "brier_baseline"    : round(eval_results[s]["brier_baseline"], 6),
        "brier_skill_score" : round(eval_results[s]["brier_skill_score"], 6),
        "n_rows_eval"       : eval_results[s]["n_rows"],
        "pos_rate_pct"      : round(eval_results[s]["pos_rate_pct"], 4),
    }
    for s in EVAL_SPLITS
}

manifest_nb04 = {
    "model": {
        "feature_cols"   : FEATURE_COLS,
        "best_iteration" : best_iter,
        "best_val_auroc" : round(best_val_auc, 6),
    },
    "training": {
        "scale_pos_weight" : round(SCALE_POS_WGT, 6),
    },
    "evaluation": {
        "per_split_metrics" : per_split_metrics,
    },
}

# ── Write artefact manifest ───────────────────────────────────────────────────
manifest_path_nb04 = MANIFEST_DIR / "model_manifest.json"
with open(manifest_path_nb04, "w") as f:
    json.dump(manifest_nb04, f, indent=2, default=str)

print(f"Manifest written to: {manifest_path_nb04}")
print(f"Active model features      : {len(FEATURE_COLS)}")
print(f"Hard exclusions inherited  : {len(HARD_EXCLUDED_FEATURES)}")
print(f"Evaluation splits          : {list(per_split_metrics.keys())}")
print(f"Target label               : {TARGET_LABEL}")

Manifest written to: /content/drive/MyDrive/master_thesis/manifests/model_manifest.json
Active model features      : 29
Hard exclusions inherited  : 8
Evaluation splits          : ['calibration', 'test_subprime', 'test_normal', 'test_covid', 'test_rate_hike']
Target label               : 60+ DPD or REO Acquisition


---

## Appendix · References

### Freddie Mac documentation

- Freddie Mac. (2026a, January). Single-family loan-level dataset frequently asked questions (FAQ) (Frequently Asked Questions). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

- Freddie Mac. (2026b, January). Single-family loan-level dataset general user guide (User Guide). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

- Freddie Mac. (2026c, January). Single-family loan-level dataset release notes (Release Notes). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

### Gradient boosting and LightGBM

- Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. The Annals of Statistics, 29(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

- Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. Advances in Neural Information Processing Systems, 30, 3146–3154. https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b7Abstract.html

- LightGBM Developers. (n.d.). *Advanced topics: Missing value handle.* LightGBM documentation. https://lightgbm.readthedocs.io/en/latest/Advanced-Topics.html

### Class imbalance and sampling design

- He, H., & Garcia, E. A. (2009). Learning from imbalanced data. IEEE Transactions on Knowledge and Data Engineering, 21(9), 1263–1284. https://doi.org/10.1109/TKDE.2008.239

### Temporal validation, leakage, and panel/event-time framing

- Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012). Leakage in data mining: Formulation, detection, and avoidance. ACM Transactions on Knowledge Discovery from Data, 6(4), Article 15, 1–21. https://doi.org/10.1145/2382577.2382579

- Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. International Journal of Forecasting, 16(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0

- Singer, J. D., & Willett, J. B. (1993). It’s about time: Using discrete-time survival analysis to study duration and the timing of events. Journal of Educational Statistics, 18(2), 155–195. https://doi.org/10.3102/10769986018002155

- Shumway, T. (2001). Forecasting bankruptcy more accurately: A simple hazard model. The Journal of Business, 74(1), 101–124. https://doi.org/10.1086/209665

### Hyperparameter optimization

- Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining, 2623–2631. https://doi.org/10.1145/3292500.3330701

- Bergstra, J. S., Bardenet, R., Bengio, Y., & Kégl, B. (2011). Algorithms for hyper-parameter optimization. In J. Shawe-Taylor, R. S. Zemel, P. L. Bartlett, F. Pereira, & K. Q. Weinberger (Eds.), Advances in neural information processing systems 24 (pp. 2546–2554). Curran Associates, Inc. https://doi.org/10.5555/2986459.2986743

- Optuna Developers. (n.d.). *Optuna documentation.* Retrieved April 27, 2026, from https://optuna.readthedocs.io/

### Predictive-performance metrics

- Fawcett, T. (2006). An introduction to ROC analysis. Pattern Recognition Letters, 27(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

- Davis, J., & Goadrich, M. (2006). The relationship between precision-recall and ROC curves. Proceedings of the 23rd International Conference on Machine Learning, 233–240. https://doi.org/10.1145/1143844.1143874

- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. Monthly Weather Review, 78(1), 1–3. https://journals.ametsoc.org/view/journals/mwre/78/1/1520-0493_1950_078_0001_vofeit_2_0_co_2.xml

### Probability calibration

- Zadrozny, B., & Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. Proceedings of the Eighth ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 694–699. https://doi.org/10.1145/775047.775151

- Niculescu-Mizil, A., & Caruana, R. (2005). Predicting good probabilities with supervised learning. Proceedings of the 22nd International Conference on Machine Learning, 625–632. https://doi.org/10.1145/1102351.1102430

- Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. In D. Precup & Y. W. Teh (Eds.), Proceedings of the 34th international conference on machine learning (pp. 1321–1330, Vol. 70). PMLR. https://proceedings.mlr.press/v70/guo17a.html

### SHAP / TreeSHAP attribution

- Lundberg, S. M., Erion, G. G., & Lee, S.-I. (2019, March 7). Consistent individualized feature attribution for tree ensembles. arXiv: 1802.03888v3 [cs.LG]. https://doi.org/10.48550/arXiv.1802.03888

- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. In I. Guyon, U. von Luxburg, S. Bengio, H. Wallach, R. Fergus, S. V. N. Vishwanathan, & R. Garnett (Eds.), Advances in neural information processing systems (pp. 4765–4774, Vol. 30). Curran Associates, Inc. https://papers.nips.cc/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html

### Conformal prediction and set-valued classification

- Vovk, V. (2012). Conditional validity of inductive conformal predictors. In S. C. H. Hoi & W. Buntine (Eds.), Proceedings of the asian conference on machine learning (pp. 475–490, Vol. 25). PMLR. https://proceedings.mlr.press/v25/vovk12.html

- Vovk, V., Gammerman, A., & Shafer, G. (2005). Algorithmic learning in a random world (1st ed.). Springer. https://doi.org/10.1007/b106715

- Romano, Y., Sesia, M., & Candès, E. J. (2020). Classification with valid and adaptive coverage. In H. Larochelle, M. Ranzato, R. Hadsell, M.-F. Balcan, & H.-T. Lin (Eds.), Advances in neural information processing systems (pp. 3581–3591, Vol. 33). Curran Associates, Inc. https://proceedings.neurips.cc/paper/2020/hash/244edd7e85dc81602b7615cd705545f5-Abstract.html

- Sadinle, M., Lei, J., & Wasserman, L. (2019). Least ambiguous set-valued classifiers with bounded error levels. Journal of the American Statistical Association, 114(525), 223–234. https://doi.org/10.1080/01621459.2017.1395341